# Lab 1: Derivatives in Practice

The limit definition of the derivative gives us the *meaning* of a derivative. The differentiation rules give us efficient ways to compute one. In this lab you will connect those two things in code: implement numerical approximations, verify the rules, and confirm that PyTorch's autograd computes exact symbolic derivatives — not finite differences.

**Sections**
1. Finite Difference Approximation
2. Secant Line → Tangent Line (Visualization)
3. Verifying the Differentiation Rules
4. The Sigmoid Derivative
5. PyTorch Autograd Verification

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print('Setup complete.')

## 1. Finite Difference Approximation

The definition of the derivative is the limit of the difference quotient:
$$f'(x) \approx \frac{f(x+h) - f(x)}{h}$$

For small $h$ this should be close to the true derivative. But $h$ can't be too small — floating-point subtraction of nearly equal numbers loses precision.

Let's explore this tradeoff for $f(x) = x^2$ at $x = 3$ (true derivative: $f'(3) = 6$).

In [ ]:
def finite_difference(f, x, h):
    """Forward difference approximation of f'(x)."""
    return (f(x + h) - f(x)) / h

f = lambda x: x**2
true_deriv = 6.0  # f'(3) = 2*3
x0 = 3.0

h_values = np.logspace(-1, -15, 60)
errors = [abs(finite_difference(f, x0, h) - true_deriv) for h in h_values]

plt.figure()
plt.loglog(h_values, errors, 'steelblue', lw=2)
plt.xlabel('Step size h')
plt.ylabel('|approximation error|')
plt.title("Finite difference error vs step size for f(x)=x² at x=3")
plt.gca().invert_xaxis()
plt.axvline(1e-8, color='coral', linestyle='--', label='floating-point noise floor')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Best approximation: {min(errors):.2e} error")
print("Notice: error decreases linearly as h shrinks, then blows up due to floating-point cancellation.")

## 2. Secant Line → Tangent Line

As $h \to 0$, the secant line through $(x_0, f(x_0))$ and $(x_0+h, f(x_0+h))$ rotates toward the tangent line at $x_0$.

Here we plot this convergence for $f(x) = x^2$ at $x_0 = 1$.

In [ ]:
f = lambda x: x**2
x0 = 1.0
x_plot = np.linspace(-0.5, 2.5, 300)

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
h_vals = [1.0, 0.5, 0.1]

for ax, h in zip(axes, h_vals):
    slope = (f(x0 + h) - f(x0)) / h
    # Secant line: passes through (x0, f(x0)) with slope = finite difference
    secant = f(x0) + slope * (x_plot - x0)
    # True tangent: f'(x0) = 2x0 = 2
    tangent = f(x0) + 2 * (x_plot - x0)

    ax.plot(x_plot, f(x_plot), 'steelblue', lw=2, label='f(x) = x²')
    ax.plot(x_plot, secant, 'coral', lw=1.5, linestyle='--', label=f'secant (h={h})')
    ax.plot(x_plot, tangent, 'forestgreen', lw=1.5, linestyle=':', label='tangent')
    ax.scatter([x0, x0 + h], [f(x0), f(x0 + h)], color='black', zorder=5)
    ax.set_xlim(-0.3, 2.3)
    ax.set_ylim(-0.5, 5)
    ax.set_title(f'h = {h}')
    ax.legend(fontsize=8)

plt.suptitle('Secant line converging to tangent as h → 0', y=1.02)
plt.tight_layout()
plt.show()

## 3. Verifying the Differentiation Rules

We compare the analytic derivative (computed by hand) to the numerical estimate. Close agreement confirms the rule is correctly applied.

| Function | Analytic $f'(x)$ | Rule used |
|---|---|---|
| $5x^3 - 2x + 7$ | $15x^2 - 2$ | Power + sum |
| $x^2 \sin x$ | $2x\sin x + x^2\cos x$ | Product |
| $(3x^2+1)^5$ | $30x(3x^2+1)^4$ | Chain |

In [ ]:
H = 1e-7  # step size for numerical check

rules = [
    ('5x³ - 2x + 7',     lambda x: 5*x**3 - 2*x + 7,      lambda x: 15*x**2 - 2),
    ('x² sin(x)',         lambda x: x**2 * np.sin(x),        lambda x: 2*x*np.sin(x) + x**2*np.cos(x)),
    ('(3x²+1)⁵',         lambda x: (3*x**2 + 1)**5,         lambda x: 30*x*(3*x**2+1)**4),
]

test_points = [0.5, 1.0, 2.0, -1.5]

print(f"{'Function':<20} {'x':>5}  {'Analytic':>12}  {'Numerical':>12}  {'Rel error':>12}")
print('-' * 70)
for name, f, df in rules:
    for x in test_points:
        analytic = df(x)
        numerical = (f(x + H) - f(x)) / H
        rel_err = abs(analytic - numerical) / (abs(analytic) + 1e-15)
        print(f"{name:<20} {x:>5.1f}  {analytic:>12.6f}  {numerical:>12.6f}  {rel_err:>12.2e}")
    print()

## 4. The Sigmoid Derivative

The sigmoid $\sigma(x) = 1/(1+e^{-x})$ has the elegant derivative $\sigma'(x) = \sigma(x)(1-\sigma(x))$.

Let's plot both and highlight the vanishing gradient zones where $\sigma' \approx 0$.

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

x = np.linspace(-6, 6, 400)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(x, sigmoid(x), 'steelblue', lw=2)
ax1.axhline(0.5, color='gray', linestyle=':', alpha=0.6)
ax1.axvline(0, color='gray', linestyle=':', alpha=0.6)
ax1.set_title('σ(x)')
ax1.set_xlabel('x')
ax1.set_ylim(-0.05, 1.05)

sd = sigmoid_derivative(x)
ax2.plot(x, sd, 'coral', lw=2)
ax2.axhline(0.25, color='gray', linestyle=':', alpha=0.6, label='max = 0.25 at x=0')
# Shade the vanishing gradient zones
ax2.fill_between(x, sd, where=(np.abs(x) > 3), alpha=0.25, color='coral', label='vanishing gradient zone')
ax2.set_title("σ'(x) = σ(x)(1 − σ(x))")
ax2.set_xlabel('x')
ax2.legend(fontsize=9)

plt.suptitle('Sigmoid and its derivative', y=1.02)
plt.tight_layout()
plt.show()

print(f"Max gradient: {sigmoid_derivative(0):.4f} (at x=0)")
print(f"Gradient at x=5: {sigmoid_derivative(5):.6f}  ← essentially zero")

## 5. PyTorch Autograd Verification

Autograd computes *exact* symbolic derivatives by walking the computation graph backward — not finite differences. Let's confirm it matches our analytic results to machine precision.

In [ ]:
checks = [
    # (label, function, analytic derivative, x_val)
    ('5x³-2x+7 at x=2',    lambda x: 5*x**3 - 2*x + 7,    15*4 - 2,          2.0),
    ('x²sin(x) at x=1',    lambda x: x**2 * torch.sin(x),  2*np.sin(1)+np.cos(1), 1.0),
    ('(3x²+1)⁵ at x=1',    lambda x: (3*x**2 + 1)**5,      30*(4**4),         1.0),
    ('σ(x) at x=0',         torch.sigmoid,                   0.25,              0.0),
]

print(f"{'Function':<25} {'Analytic':>12}  {'Autograd':>12}  {'Match?':>8}")
print('-' * 65)
for label, f, analytic, xv in checks:
    x = torch.tensor(xv, requires_grad=True)
    y = f(x)
    y.backward()
    ag = x.grad.item()
    match = abs(ag - analytic) < 1e-5
    print(f"{label:<25} {analytic:>12.6f}  {ag:>12.6f}  {'✓' if match else '✗':>8}")